In [1]:
import json
import os
import boto3
import pandas as pd
pd.set_option("display.max_columns",50)
from io import StringIO

In [2]:
aws_access_key = os.getenv("AWS_ACCESS_KEY")
aws_secret_key = os.getenv("AWS_SECRET_KEY")

In [3]:
s3=boto3.client("s3", aws_access_key_id=aws_access_key, aws_secret_access_key=aws_secret_key)

In [4]:
bucket = "edem1550-chicago-taxi-bb"

dim_community_areas_path = "transformed_data/dim_community_areas/dim_community_areas.csv"
dim_company_path = "transformed_data/dim_company/dim_company.csv"
dim_date_path = "transformed_data/dim_date/dim_date.csv"
dim_payment_type_path = "transformed_data/dim_payment_type/dim_payment_type.csv"
dim_weather_path = "transformed_data/dim_weather/"
fact_taxi_trips_path = "transformed_data/fact_taxi_trips/"

In [5]:
def read_file_from_s3(
    s3, 
    bucket:str,
    key:str, 
    file_format:str="csv"
):
    
    """
    Reads a file from given S3 bucket
    Args:
        s3 (boto3.client): S3 client
        bucket (str): S3 bucket name where the file stored
        key (str): Path within the S3 bucket
        file_format (str, optional): File format. Defaults to "csv". Can be also "json"

    Returns:
        pandas.DataFrame: Dataframe
    """

    BUCKET= bucket
    response= s3.get_object(Bucket=BUCKET, Key=key)
    content= response["Body"].read().decode("utf-8")
    if file_format == "csv":
        return pd.read_csv(StringIO(content))
    elif file_format == "json":
        return json.loads(content)
    else:
        raise ValueError("File format not supported, use 'csv' or 'json' ")



In [6]:
dim_community_areas = read_file_from_s3(s3, bucket, dim_community_areas_path)
dim_company = read_file_from_s3(s3, bucket, dim_company_path)
dim_date = read_file_from_s3(s3, bucket, dim_date_path)
dim_payment_type = read_file_from_s3(s3, bucket, dim_payment_type_path)

In [7]:
dim_payment_type.head()

,payment_type_id,payment_type
0,1,Cash
1,2,Mobile
2,3,Credit Card
3,4,No Charge
4,5,Prcard


In [8]:
fact_taxi_trips_list = []
dim_weather_list = []

In [9]:
for file in s3.list_objects(Bucket=bucket, Prefix=fact_taxi_trips_path)["Contents"]:
    taxi_key = file["Key"]
    taxi_raw_file_name=file["Key"].split("/")[-1]
    if taxi_raw_file_name.split(".")[-1] =="csv":
        daily_file_name = fact_taxi_trips_path + taxi_raw_file_name
        taxi_trip_daily = read_file_from_s3(s3, bucket, daily_file_name)
        fact_taxi_trips_list.append(taxi_trip_daily)
        print(f"{taxi_raw_file_name} has been added")


taxi_2025-09-23.csv has been added
taxi_2025-09-24.csv has been added
taxi_2025-09-25.csv has been added
taxi_2025-09-26.csv has been added
taxi_2025-09-27.csv has been added
taxi_2025-09-28.csv has been added
taxi_2025-09-29.csv has been added
taxi_2025-09-30.csv has been added
taxi_2025-10-01.csv has been added
taxi_2025-10-02.csv has been added
taxi_2025-10-03.csv has been added
taxi_2025-10-04.csv has been added
taxi_2025-10-05.csv has been added


In [10]:
fact_taxi_trips = pd.concat(fact_taxi_trips_list, ignore_index=True)

In [11]:
dim_weather_list = []

In [12]:
for file in s3.list_objects(Bucket=bucket, Prefix=dim_weather_path)["Contents"]:
    weather_key = file["Key"]
    weather_raw_file_name=file["Key"].split("/")[-1]
    if weather_raw_file_name.split(".")[-1] =="csv":
        daily_file_name = dim_weather_path + weather_raw_file_name
        weather_daily = read_file_from_s3(s3, bucket, weather_key)   
        dim_weather_list.append(weather_daily)
        print(f"{weather_raw_file_name} has been added")

weather_2025-09-23.csv has been added
weather_2025-09-24.csv has been added
weather_2025-09-25.csv has been added
weather_2025-09-26.csv has been added
weather_2025-09-27.csv has been added
weather_2025-09-28.csv has been added
weather_2025-09-29.csv has been added
weather_2025-09-30.csv has been added
weather_2025-10-01.csv has been added
weather_2025-10-02.csv has been added
weather_2025-10-03.csv has been added
weather_2025-10-04.csv has been added
weather_2025-10-05.csv has been added


In [13]:
dim_weather = pd.concat(dim_weather_list, ignore_index=True)

dim_weather.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   datetime       312 non-null    object 
 1   temperature    312 non-null    float64
 2   wind_speed     312 non-null    float64
 3   rain           312 non-null    float64
 4   precipitation  312 non-null    float64
dtypes: float64(4), object(1)
memory usage: 12.3+ KB


### Create datamodel

In [14]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips, dim_weather, left_on="datetime_for_weather", right_on="datetime")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["datetime", "datetime_for_weather"])

In [15]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_company, left_on="company_id", right_on="company_id")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["company_id"])

In [16]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_payment_type, left_on="payment_type_id", right_on="payment_type_id")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["payment_type_id"])

In [17]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_community_areas, left_on="pickup_community_area_id", right_on="area_code")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["pickup_community_area_id","area_code",])
fact_taxi_trips_full.rename(columns={"community_area":"pickup_community_area_name"}, inplace=True)

In [18]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_community_areas, left_on="dropoff_community_area_id", right_on="area_code")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["dropoff_community_area_id","area_code",])
fact_taxi_trips_full.rename(columns={"community_area":"dropoff_community_area_name"}, inplace=True)

In [19]:
dim_date["date"] = pd.to_datetime(dim_date["date"])
fact_taxi_trips_full["trip_start_timestamp"] = pd.to_datetime(fact_taxi_trips_full["trip_start_timestamp"])

fact_taxi_trips_full["trip_start_date"] = pd.to_datetime(fact_taxi_trips_full["trip_start_timestamp"].dt.date)


In [ ]:
fact_taxi_trips_full = pd.merge(fact_taxi_trips_full, dim_date, left_on="trip_start_date", right_on="date")
fact_taxi_trips_full = fact_taxi_trips_full.drop(columns=["trip_start_date","date"])
fact_taxi_trips_full.info()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,temperature,wind_speed,rain,precipitation,company,payment_type,pickup_community_area_name,dropoff_community_area_name,year,month,day,day_of_week,is_weekend
0,93446c9c1098f23e505164659baba263728e620d,02ef8f01232b1b1828f4e5e1b8e8a85cd71b67c449afaf...,2025-09-23 23:45:00,2025-09-24T00:15:00.000,1134,11.60,30.75,0.0,0.0,0.0,30.75,41.763247,-87.616134,41.899602,-87.633308,20.2,12.2,0.0,0.0,City Service,Prcard,Greater Grand Crossing,Near North Side,2025,9,23,2,False
1,928acc35835559e18bfd640170fc2632f18c1b10,6544c8051894de482dd3253d911c3e3d713b43beb93de2...,2025-09-23 23:45:00,2025-09-24T00:00:00.000,726,3.49,11.75,8.0,0.0,0.0,20.25,41.944227,-87.655998,41.922761,-87.699155,20.2,12.2,0.0,0.0,City Service,Credit Card,Lake View,Logan Square,2025,9,23,2,False
2,8f6f96baf9284ac1acd4c2eee2f56dd8ac8d0e7d,f509c57d2f0b196f54d9b751ce2a1b6e956f378841b1e8...,2025-09-23 23:45:00,2025-09-23T23:45:00.000,4,0.00,25.00,2.0,0.0,0.0,27.50,41.878866,-87.625192,41.878866,-87.625192,20.2,12.2,0.0,0.0,City Service,Credit Card,Loop,Loop,2025,9,23,2,False
3,85a4e27c77425d5a8f6d0253585c98f083ddfec8,8d9a2218e0a2c8ae95ec61ac346b42b4e2fe21c484c89d...,2025-09-23 23:45:00,2025-09-24T00:00:00.000,610,9.16,24.25,0.0,0.0,0.0,24.25,41.706588,-87.623367,41.835118,-87.618678,20.2,12.2,0.0,0.0,Globe Taxi,Prcard,Roseland,Douglas,2025,9,23,2,False
4,84db5e3c929af11688df7216e13402a95bf838f6,b5e2695a2f44b9bce7a0a86148ac418802f0067be1f6d4...,2025-09-23 23:45:00,2025-09-23T23:45:00.000,10,0.08,3.25,0.0,0.0,0.0,3.25,41.979071,-87.903040,41.979071,-87.903040,20.2,12.2,0.0,0.0,Sun Taxi,Cash,O'Hare[11],O'Hare[11],2025,9,23,2,False


In [21]:
fact_taxi_trips_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252225 entries, 0 to 252224
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   trip_id                      252225 non-null  object        
 1   taxi_id                      252225 non-null  object        
 2   trip_start_timestamp         252225 non-null  datetime64[ns]
 3   trip_end_timestamp           252225 non-null  object        
 4   trip_seconds                 252225 non-null  int64         
 5   trip_miles                   252225 non-null  float64       
 6   fare                         252225 non-null  float64       
 7   tips                         252225 non-null  float64       
 8   tolls                        252225 non-null  float64       
 9   extras                       252225 non-null  float64       
 10  trip_total                   252225 non-null  float64       
 11  pickup_centroid_latitude  